In [3]:
from db import (
 get_connection
)
import sqlite3

In [4]:
import os
import sqlite3

# 1. Check if the file actually exists where you think it is
db_file = "jobs.db"
if os.path.exists(db_file):
    print(f"✅ Found {db_file} ({os.path.getsize(db_file)} bytes)")
else:
    print(f"❌ Could not find {db_file} in {os.getcwd()}")

# 2. Test the connection and check table names
try:
    with get_connection() as conn:
        cursor = conn.cursor()
        
        # Check if the 'jobs' table exists
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='jobs';")
        if not cursor.fetchone():
            print("❌ The table 'jobs' does not exist in this database.")
        else:
            # Check the row count
            count = cursor.execute("SELECT COUNT(*) FROM jobs").fetchone()[0]
            print(f"📊 Row count in 'jobs' table: {count}")
            
            if count > 0:
                # Try to fetch one row just to be sure
                sample = cursor.execute("SELECT title, description FROM jobs LIMIT 1").fetchone()
                print(f"📝 Sample Title: {sample[0]}")
                print(f"📏 Description Length: {len(sample[1]) if sample[1] else 0}")
            else:
                print("⚠️ The 'jobs' table is empty. Did you run your Adzuna scraper yet?")
                
except Exception as e:
    print(f"🔥 Database Error: {e}")

✅ Found jobs.db (110592 bytes)
📊 Row count in 'jobs' table: 58
📝 Sample Title: Research Intern - Computational Social Science
📏 Description Length: 500


**Testing beautiful soup**

In [8]:
import sqlite3
import requests
import time
import re

# Use your existing get_connection or define it here
def get_connection():
    return sqlite3.connect("jobs.db")

def hydrate_all_jobs():
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
    }

    # 1. Get all truncated jobs
    with get_connection() as conn:
        conn.row_factory = sqlite3.Row
        jobs = conn.execute(
            "SELECT id, url, title, company FROM jobs WHERE LENGTH(description) <= 505"
        ).fetchall()

    print(f"🔍 Found {len(jobs)} truncated jobs. Starting hydration...\n")

    success = 0
    failed = 0

    for job in jobs:
        print(f"🌐 Fetching: {job['title']} @ {job['company']}...")
        
        try:
            # We follow redirects because Adzuna uses them
            response = requests.get(job['url'], headers=headers, timeout=15, allow_redirects=True)
            
            if response.status_code == 200:
                # Basic text extraction without BeautifulSoup (since it's a notebook test)
                # This grabs the bulk text and cleans up some HTML tags via Regex
                raw_html = response.text
                
                # Simple Regex to strip HTML tags if you don't have BS4 installed yet
                clean_text = re.sub('<[^<]+?>', '', raw_html)
                
                # Clean up whitespace
                lines = [l.strip() for l in clean_text.split('\n') if len(l.strip()) > 30]
                final_text = "\n".join(lines)

                if len(final_text) > 505:
                    with get_connection() as conn:
                        conn.execute(
                            "UPDATE jobs SET description = ? WHERE id = ?", 
                            (final_text, job['id'])
                        )
                        conn.commit()
                    print(f"✅ Updated! New length: {len(final_text)}")
                    success += 1
                else:
                    print(f"⚠️  Page loaded but no long text found ({len(final_text)} chars).")
                    failed += 1
            else:
                print(f"❌ Failed: HTTP {response.status_code}")
                failed += 1

        except Exception as e:
            print(f"🔥 Error: {e}")
            failed += 1
        
        # Polite delay
        time.sleep(1.5)

    print(f"\n--- DONE ---\nSuccess: {success}\nFailed: {failed}")

# Execute in your notebook
hydrate_all_jobs()

🔍 Found 58 truncated jobs. Starting hydration...

🌐 Fetching: Research Intern - Computational Social Science @ Microsoft Corporation...
❌ Failed: HTTP 403
🌐 Fetching: Intern, Data Engineering @ Octaura...
❌ Failed: HTTP 403
🌐 Fetching: Data Science Intern - Summer 2026 @ Greater New York Mutual Insurance Company...
✅ Updated! New length: 21886
🌐 Fetching: Intern - Data Science/Analyst (Hybrid in NY) @ Credibly...
✅ Updated! New length: 35327
🌐 Fetching: Summer 2026 Internship: Data Science - Long Island City, NY @ dsm-firmenich...
❌ Failed: HTTP 403
🌐 Fetching: Data Analyst Intern @ RTW Investments...
✅ Updated! New length: 27258
🌐 Fetching: Customer Data Analyst Intern @ Topcon Healthcare...
✅ Updated! New length: 33959
🌐 Fetching: Intern - Data Analyst @ Coherent...
✅ Updated! New length: 37468
🌐 Fetching: Internship - Jr. Analyst, CRM Data & Analytics @ CentralReach...
✅ Updated! New length: 33873
🌐 Fetching: Sector Business Analyst Intern @ Vector Solutions...
✅ Updated! New length

In [10]:
import sqlite3

def cleanup_short_jobs(threshold=1000, dry_run=True):
    with get_connection() as conn:
        conn.row_factory = sqlite3.Row
        
        # 1. Identify the rows
        to_delete = conn.execute(
            "SELECT id, title, company, LENGTH(description) as length FROM jobs WHERE LENGTH(description) < ?", 
            (threshold,)
        ).fetchall()
        
        if not to_delete:
            print(f"✨ No jobs found with description length under {threshold} characters.")
            return

        print(f"🗑️ Found {len(to_delete)} jobs to delete (Length < {threshold} chars):")
        for job in to_delete:
            print(f"  - [{job['length']} chars] {job['title']} @ {job['company']}")

        # 2. Perform deletion if not a dry run
        if not dry_run:
            confirm = input(f"\n⚠️ Are you sure you want to delete these {len(to_delete)} rows? (y/n): ")
            if confirm.lower() == 'y':
                conn.execute("DELETE FROM jobs WHERE LENGTH(description) < ?", (threshold,))
                conn.commit()
                print(f"\n✅ Successfully deleted {len(to_delete)} rows.")
            else:
                print("\n❌ Deletion cancelled.")
        else:
            print(f"\nℹ️ This was a DRY RUN. No rows were deleted. Set 'dry_run=False' to execute.")

# --- RUN THE CLEANUP ---
# Change dry_run=False when you are ready to actually delete them
cleanup_short_jobs(threshold=1000, dry_run=False)

🗑️ Found 38 jobs to delete (Length < 1000 chars):
  - [500 chars] Research Intern - Computational Social Science @ Microsoft Corporation
  - [500 chars] Intern, Data Engineering @ Octaura
  - [500 chars] Summer 2026 Internship: Data Science - Long Island City, NY @ dsm-firmenich
  - [500 chars] Business Development Analyst Intern @ Scientech Research
  - [500 chars] 2026 AI Engineer Intern, Long/Short Equities @ Point72
  - [500 chars] Product Management Internships - Academic Year @ NBC Universal
  - [500 chars] Manager-Data Science @ American Express
  - [500 chars] Manager, Data Science - AI Foundations @ Capital One
  - [500 chars] Senior Manager, Data Science - AI Foundations @ Capital One
  - [500 chars] Manager, Data Science - AI for Data @ Capital One
  - [500 chars] Marketing Data Science Consultant @ US Tech Solutions
  - [500 chars] Head of Data Science & AI @ Janus Henderson Investors
  - [500 chars] Global Head, Data Science @ S&P Global
  - [500 chars] Senior Associate, D